In [1]:
# Personal parameters
SID4 = 640
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A, CLS_B = (SID4 % 10), ((SID4 // 10) % 10)

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)

SID4: 640
SEED: 640
SLICE: 640
HP_ID: 4
CLS_A: 0
CLS_B: 4


# Task 1: GPT-Style LLM from Scratch

## 1.1 Data Preprocessing

### 1.1.1 Load the Dataset and Inspect the Dataset

TinyStories is read one story at a time using `<|endoftext|>` as the
story delimiter. Empty entries are skipped. Case, punctuation, spaces,
and internal line breaks are preserved.

Each character is one token. Story delimiters are removed during
reading; two newline characters will separate stories in the encoded
text stream.

In [2]:
from pathlib import Path
from itertools import islice

# Locate shared data using the repository structure.
for folder in [Path.cwd(), *Path.cwd().parents]:
    data_dir = folder / "task1_llm" / "data"
    if data_dir.is_dir():
        break
else:
    raise FileNotFoundError("Cannot find task1_llm/data")

train_path = data_dir / "TinyStories-train.txt"
valid_path = data_dir / "TinyStories-valid.txt"


def iter_stories(path):
    """Yield one non-empty story at a time."""
    lines = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip() == "<|endoftext|>":
                story = "".join(lines).strip()
                if story:
                    yield story
                lines = []
            else:
                lines.append(line)

    story = "".join(lines).strip()
    if story:
        yield story


# Inspect both files.
for path in (train_path, valid_path):
    if not path.is_file():
        raise FileNotFoundError(path.name)

    print(f"{path.name}: {path.stat().st_size / 1e6:.1f} MB")

# Preview only; formal sampling follows in Section 1.1.2.
preview_stories = list(islice(iter_stories(train_path), 2))

for index, story in enumerate(preview_stories, start=1):
    print(f"\nStory {index} | {len(story):,} characters")
    print(story)

TinyStories-train.txt: 1924.3 MB
TinyStories-valid.txt: 19.4 MB

Story 1 | 699 characters
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.
Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."
Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.

Story 2 | 703 characters
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and st

### 1.1.2 Prepare Training and Validation Sets

The required 100K/10K sizes are interpreted as story counts. Independently sample 100,000 training stories and 10,000 validation stories while preserving the official train–validation boundary.

Reservoir sampling with fixed seeds makes the selection reproducible. SHA-256 hashes identify exact duplicates in the parsed story text.
Duplicate stories are skipped, and selected training stories are excluded from validation sampling.

Sampling scans each source file but retains only the selected story texts and hashes of previously encountered stories.

In [3]:
import hashlib
import random

TRAIN_SIZE = 100_000
VALID_SIZE = 10_000
TRAIN_SEED = SEED
VALID_SEED = SEED + 1


def story_hash(story):
    return hashlib.sha256(story.encode("utf-8")).digest()


def sample_unique_stories(path, size, seed, excluded_hashes=None):
    """Uniformly sample distinct eligible stories."""
    rng = random.Random(seed)
    excluded = excluded_hashes if excluded_hashes is not None else set()

    seen = set()
    sample = []
    eligible_count = 0

    for story in iter_stories(path):
        digest = story_hash(story)

        if digest in seen or digest in excluded:
            continue

        seen.add(digest)
        eligible_count += 1

        if len(sample) < size:
            sample.append(story)
        else:
            position = rng.randrange(eligible_count)
            if position < size:
                sample[position] = story

    if len(sample) != size:
        raise ValueError(
            f"Requested {size:,} stories; found {eligible_count:,} eligible."
        )

    rng.shuffle(sample)
    return sample


# Independently select the two subsets.
train_stories = sample_unique_stories(
    train_path, TRAIN_SIZE, TRAIN_SEED
)
train_hashes = {story_hash(story) for story in train_stories}
print("Training sampling complete.")

valid_stories = sample_unique_stories(
    valid_path, VALID_SIZE, VALID_SEED,
    excluded_hashes=train_hashes,
)
valid_hashes = {story_hash(story) for story in valid_stories}

# Check sizes, exact duplicates, and overlap.
train_duplicates = len(train_stories) - len(train_hashes)
valid_duplicates = len(valid_stories) - len(valid_hashes)
overlap = len(train_hashes & valid_hashes)

assert len(train_stories) == TRAIN_SIZE
assert len(valid_stories) == VALID_SIZE
assert train_duplicates == valid_duplicates == overlap == 0

print(f"\nTraining stories: {len(train_stories):,}")
print(f"Validation stories: {len(valid_stories):,}")
print(f"Training duplicates: {train_duplicates}")
print(f"Validation duplicates: {valid_duplicates}")
print(f"Train–validation overlap: {overlap}")
print(f"Sampling seeds: train={TRAIN_SEED}, validation={VALID_SEED}")

print("\nFirst selected training story:")
print(train_stories[0][:300])

Training sampling complete.

Training stories: 100,000
Validation stories: 10,000
Training duplicates: 0
Validation duplicates: 0
Train–validation overlap: 0
Sampling seeds: train=640, validation=641

First selected training story:
Once upon a time there was a little girl who was very grumpy. She scrunched up her face and complained a lot. One day, her mommy told her to take a break and relax. The little girl said no. She wanted to keep doing what she was doing. But her mommy said it was time to take a break, so the little gir


### 1.1.3 Character-Level Tokenization and Integer Encoding

Each character is one token. Two newline characters separate stories in each subset's text stream.

Build `char_to_idx` and `idx_to_char` from training characters only. Assign IDs to sorted characters for deterministic encoding. Reserve ID 0 for `<UNK>` to handle unseen validation characters; validation data does not modify the vocabulary.

Encode all selected stories as NumPy int32 arrays. Display a short example and verify that training text can be reconstructed from its IDs.

In [4]:
import numpy as np


def character_tokens(stories):
    """Yield every character, inserting two newlines between stories."""
    for index, story in enumerate(stories):
        if index > 0:
            yield from "\n\n"
        yield from story


# Build the vocabulary using training data only.
characters = sorted(set(character_tokens(train_stories)))

char_to_idx = {"<UNK>": 0}
char_to_idx.update({
    character: index
    for index, character in enumerate(characters, start=1)
})
idx_to_char = {
    index: character for character, index in char_to_idx.items()
}


def encode_stories(stories):
    return np.fromiter(
        (
            char_to_idx.get(character, char_to_idx["<UNK>"])
            for character in character_tokens(stories)
        ),
        dtype=np.int32,
    )


# Encode ALL selected stories, not just the displayed example.
train_ids = encode_stories(train_stories)
valid_ids = encode_stories(valid_stories)

# Verify dictionary consistency and the absence of unknown training tokens.
assert all(idx_to_char[index] == char
           for char, index in char_to_idx.items())
assert not np.any(train_ids == char_to_idx["<UNK>"])

# Inspect a short example from the actual encoded training stream.
example_text = train_stories[0][:80]
example_ids = train_ids[:len(example_text)]
reconstructed = "".join(idx_to_char[int(i)] for i in example_ids)

assert reconstructed == example_text

print(f"Vocabulary size (including <UNK>): {len(char_to_idx):,}")
print(f"Training tokens: {len(train_ids):,}")
print(f"Validation tokens: {len(valid_ids):,}")
print(f"Validation <UNK> rate: {np.mean(valid_ids == 0):.6%}")

print("\nFirst 15 vocabulary entries:")
print([(repr(idx_to_char[i]), i)
       for i in range(min(15, len(idx_to_char)))])

print("\nOriginal text:", repr(example_text))
print("Character tokens:", list(example_text))
print("Integer IDs:", example_ids.tolist())
print("Reconstructed text:", repr(reconstructed))

Vocabulary size (including <UNK>): 104
Training tokens: 88,792,995
Validation tokens: 8,693,309
Validation <UNK> rate: 0.000023%

First 15 vocabulary entries:
[("'<UNK>'", 0), ("'\\t'", 1), ("'\\n'", 2), ("' '", 3), ("'!'", 4), ('\'"\'', 5), ("'#'", 6), ("'$'", 7), ("'%'", 8), ("'&'", 9), ('"\'"', 10), ("'('", 11), ("')'", 12), ("'*'", 13), ("'+'", 14)]

Original text: 'Once upon a time there was a little girl who was very grumpy. She scrunched up h'
Character tokens: ['O', 'n', 'c', 'e', ' ', 'u', 'p', 'o', 'n', ' ', 'a', ' ', 't', 'i', 'm', 'e', ' ', 't', 'h', 'e', 'r', 'e', ' ', 'w', 'a', 's', ' ', 'a', ' ', 'l', 'i', 't', 't', 'l', 'e', ' ', 'g', 'i', 'r', 'l', ' ', 'w', 'h', 'o', ' ', 'w', 'a', 's', ' ', 'v', 'e', 'r', 'y', ' ', 'g', 'r', 'u', 'm', 'p', 'y', '.', ' ', 'S', 'h', 'e', ' ', 's', 'c', 'r', 'u', 'n', 'c', 'h', 'e', 'd', ' ', 'u', 'p', ' ', 'h']
Integer IDs: [47, 75, 64, 66, 3, 82, 77, 76, 75, 3, 62, 3, 81, 70, 74, 66, 3, 81, 69, 66, 79, 66, 3, 84, 62, 80, 3, 62, 3, 73,

### 1.1.4 Create Fixed-Length Input–Target Sequences

Use a context length of 256 characters.
Each example requires 257 consecutive encoded characters:

- Input: the first 256 characters.
- Target: the last 256 characters, shifted forward by one position.

Window starts advance by 256 characters. An incomplete final window is discarded. Windows may cross the newline-separated story boundaries, but never cross between training and validation subsets.

The dataset creates tensor pairs on demand, avoiding storage of every window as a separate tensor. Story counts and sequence counts are reported separately.

In [5]:
import torch
from torch.utils.data import Dataset

CONTEXT_LENGTH = 256


class NextCharacterDataset(Dataset):
    def __init__(self, token_ids, context_length):
        if context_length < 1 or len(token_ids) <= context_length:
            raise ValueError("Invalid context length or insufficient tokens.")

        self.token_ids = token_ids
        self.context_length = context_length

    def __len__(self):
        return (len(self.token_ids) - 1) // self.context_length

    def __getitem__(self, index):
        if not 0 <= index < len(self):
            raise IndexError(index)

        start = index * self.context_length
        chunk = self.token_ids[start:start + self.context_length + 1]

        # Convert only this window to the integer type used by PyTorch.
        chunk = torch.from_numpy(chunk.astype(np.int64))
        return chunk[:-1], chunk[1:]


train_dataset = NextCharacterDataset(train_ids, CONTEXT_LENGTH)
valid_dataset = NextCharacterDataset(valid_ids, CONTEXT_LENGTH)

# Check both the first and last examples in each subset.
for dataset in (train_dataset, valid_dataset):
    for index in (0, len(dataset) - 1):
        x, y = dataset[index]
        start = index * CONTEXT_LENGTH

        assert x.shape == y.shape == (CONTEXT_LENGTH,)
        assert x.dtype == y.dtype == torch.long
        assert torch.equal(x[1:], y[:-1])
        assert x[0].item() == int(dataset.token_ids[start])
        assert y[-1].item() == int(
            dataset.token_ids[start + CONTEXT_LENGTH]
        )

# Inspect an actual training example.
x, y = train_dataset[0]

print(f"Training stories: {len(train_stories):,}")
print(f"Validation stories: {len(valid_stories):,}")
print(f"Training sequences: {len(train_dataset):,}")
print(f"Validation sequences: {len(valid_dataset):,}")
print(f"Input shape: {tuple(x.shape)}")
print(f"Target shape: {tuple(y.shape)}")

print("\nFirst 20 input IDs:", x[:20].tolist())
print("First 20 target IDs:", y[:20].tolist())

print("\nInput text preview:")
print(repr("".join(idx_to_char[i] for i in x[:80].tolist())))

print("Target text preview (shifted by one character):")
print(repr("".join(idx_to_char[i] for i in y[:80].tolist())))

print("\nAll preprocessing checks passed.")

Training stories: 100,000
Validation stories: 10,000
Training sequences: 346,847
Validation sequences: 33,958
Input shape: (256,)
Target shape: (256,)

First 20 input IDs: [47, 75, 64, 66, 3, 82, 77, 76, 75, 3, 62, 3, 81, 70, 74, 66, 3, 81, 69, 66]
First 20 target IDs: [75, 64, 66, 3, 82, 77, 76, 75, 3, 62, 3, 81, 70, 74, 66, 3, 81, 69, 66, 79]

Input text preview:
'Once upon a time there was a little girl who was very grumpy. She scrunched up h'
Target text preview (shifted by one character):
'nce upon a time there was a little girl who was very grumpy. She scrunched up he'

All preprocessing checks passed.


### 1.1.5 Save Preprocessing Artifacts

Save the encoded arrays, vocabulary, and ordered story hashes under
`Yuyao_Ding/data_processed/`. Save the preprocessing configuration under
`configs/` and an artifact manifest under `reproducibility/manifests/Yuyao_Ding/`.
All recorded paths are relative to the repository root.

Run Sections 1.1.1–1.1.4 in order before this cell. Before saving, verify
that the selected stories reproduce under the recorded sampling seeds and
that the encoded arrays match those stories. This prevents saving stale
notebook variables after changing a seed.

The manifest records source-file and output-file SHA-256 checksums,
package versions, counts, and the save time. Reload the saved arrays and
vocabulary to check that they match the in-memory data. These artifacts
contain preprocessing results, not model weights or training results.


In [6]:
import json
import platform
from datetime import datetime, timezone

repo_root = data_dir.parent.parent
member_dir = data_dir.parent / "Yuyao_Ding"
artifact_dir = member_dir / "data_processed"
config_dir = member_dir / "configs"
manifest_dir = repo_root / "reproducibility" / "manifests" / "Yuyao_Ding"


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path, value):
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


# Check provenance before writing: changing a seed requires new data.
ordered_hashes = {}
for split, path, stories, ids, size, seed in [
    ("train", train_path, train_stories, train_ids, TRAIN_SIZE, TRAIN_SEED),
    ("validation", valid_path, valid_stories, valid_ids, VALID_SIZE, VALID_SEED),
]:
    hashes = [story_hash(story).hex() for story in stories]
    exclusions = {story_hash(s) for s in train_stories} if split == "validation" else None
    reproduced = sample_unique_stories(path, size, seed, exclusions)
    if hashes != [story_hash(story).hex() for story in reproduced]:
        raise ValueError(f"Stale {split} stories: rerun Sections 1.1.2–1.1.4.")
    if not np.array_equal(ids, encode_stories(stories)):
        raise ValueError(f"Stale {split} encoding: rerun Sections 1.1.3–1.1.4.")
    ordered_hashes[split] = hashes
    del reproduced

for dataset, ids in [(train_dataset, train_ids), (valid_dataset, valid_ids)]:
    if dataset.token_ids is not ids or dataset.context_length != CONTEXT_LENGTH:
        raise ValueError("Stale sequence dataset: rerun Section 1.1.4.")

for directory in (artifact_dir, config_dir, manifest_dir):
    directory.mkdir(parents=True, exist_ok=True)

config = {
    "train_story_count": TRAIN_SIZE,
    "validation_story_count": VALID_SIZE,
    "train_seed": TRAIN_SEED,
    "validation_seed": VALID_SEED,
    "context_length": CONTEXT_LENGTH,
    "sequence_stride": CONTEXT_LENGTH,
    "size_unit": "stories",
    "sampling": "reservoir over unique parsed stories; preserve official split",
    "reader": "UTF-8; standalone <|endoftext|> delimiter; strip story edges; skip empty",
    "tokenization": "Unicode characters; training-only sorted vocabulary",
    "story_separator": "\n\n",
    "unknown_id": char_to_idx["<UNK>"],
    "array_dtype": "int32",
    "incomplete_window": "discard",
}

np.save(artifact_dir / "train_ids.npy", train_ids, allow_pickle=False)
np.save(artifact_dir / "valid_ids.npy", valid_ids, allow_pickle=False)
# A list preserves ID order without JSON converting integer keys to strings.
vocabulary = [idx_to_char[i] for i in range(len(idx_to_char))]
write_json(artifact_dir / "vocabulary.json", vocabulary)
write_json(artifact_dir / "split_story_hashes.json", ordered_hashes)
write_json(config_dir / "preprocessing.json", config)

manifest = {
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "config": (config_dir / "preprocessing.json").relative_to(repo_root).as_posix(),
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "torch": str(torch.__version__),
    },
    "sources": {
        path.relative_to(repo_root).as_posix(): {
            "bytes": path.stat().st_size, "sha256": file_sha256(path)
        }
        for path in (train_path, valid_path)
    },
    "statistics": {
        "vocabulary_size": len(vocabulary),
        "train_tokens": len(train_ids),
        "validation_tokens": len(valid_ids),
        "train_sequences": len(train_dataset),
        "validation_sequences": len(valid_dataset),
        "validation_unknown_tokens": int(np.count_nonzero(valid_ids == 0)),
    },
    "artifacts": {
        path.relative_to(repo_root).as_posix(): {
            "bytes": path.stat().st_size, "sha256": file_sha256(path)
        }
        for path in [
            artifact_dir / "train_ids.npy", artifact_dir / "valid_ids.npy",
            artifact_dir / "vocabulary.json", artifact_dir / "split_story_hashes.json",
            config_dir / "preprocessing.json",
        ]
    },
}

# Verify the saved artifacts by loading them back.
for name, expected in [("train_ids.npy", train_ids), ("valid_ids.npy", valid_ids)]:
    restored = np.load(artifact_dir / name, mmap_mode="r", allow_pickle=False)
    assert restored.dtype == expected.dtype
    assert np.array_equal(restored, expected)
    del restored

restored_vocab = json.loads((artifact_dir / "vocabulary.json").read_text(encoding="utf-8"))
assert {i: char for i, char in enumerate(restored_vocab)} == idx_to_char
assert {char: i for i, char in enumerate(restored_vocab)} == char_to_idx
assert json.loads((artifact_dir / "split_story_hashes.json").read_text()) == ordered_hashes
assert json.loads((config_dir / "preprocessing.json").read_text()) == config

write_json(manifest_dir / "preprocessing_manifest.json", manifest)
print(f"Verified sampling seeds: train={TRAIN_SEED}, validation={VALID_SEED}")
for relative_path in manifest["artifacts"]:
    print(f"Saved and verified: {relative_path}")
print("Manifest:", (manifest_dir / "preprocessing_manifest.json").relative_to(repo_root))
print("All preprocessing artifacts saved and verified.")


Verified sampling seeds: train=640, validation=641
Saved and verified: task1_llm/Yuyao_Ding/data_processed/train_ids.npy
Saved and verified: task1_llm/Yuyao_Ding/data_processed/valid_ids.npy
Saved and verified: task1_llm/Yuyao_Ding/data_processed/vocabulary.json
Saved and verified: task1_llm/Yuyao_Ding/data_processed/split_story_hashes.json
Saved and verified: task1_llm/Yuyao_Ding/configs/preprocessing.json
Manifest: reproducibility/manifests/Yuyao_Ding/preprocessing_manifest.json
All preprocessing artifacts saved and verified.
